## News collection (Alpha Vantage)

Replaces the previous Finnhub-based collection. `company_news` on Finnhub's free
tier only has ~1 year of archive, so windows before mid-2025 were silently
returning zero articles (no exception raised) and getting imputed to neutral
(0.0) sentiment downstream — effectively fabricating sentiment for ~80% of rows.

Alpha Vantage's `NEWS_SENTIMENT` endpoint archives back to 2021+ for our
tickers, and is queried per-ticker over a date range (not per earnings
window). Running on the **premium tier** (75 requests/min, no daily cap), so
this notebook:

1. Fetches all news per ticker once, paginating with `time_from`/`time_to`,
   and caches raw results to disk so reruns resume instead of re-fetching.
2. Stops cleanly (without losing progress) if the API returns a rate-limit
   or error response — rerun the fetch cell to continue.
3. Slices the cached articles into each earnings window locally.
4. Explicitly reports windows with zero articles, and distinguishes
   "genuinely no news" from "backfill for this ticker isn't finished yet" —
   the previous pipeline conflated these, which is what hid the coverage gap.

Note: some tickers were partially backfilled on the free tier before
upgrading — the on-disk cache in `av_news_cache/` doesn't care which API key
fetched it, so reruns pick up exactly where they left off regardless of the
tier switch.

Output format (`ticker`, `earnings_date`, `full_text`) is unchanged, so the
Colab FinBERT step and `src/features.py` downstream need no changes.


In [1]:
import os
import json
import time
from datetime import datetime, timedelta

import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()


True

In [2]:
AV_KEY = os.environ["AV_API_KEY"]
AV_URL = "https://www.alphavantage.co/query"

DATA_DIR = "../data/raw"
BASE_CSV = os.path.join(DATA_DIR, "earnings_base.csv")
CACHE_DIR = os.path.join(DATA_DIR, "av_news_cache")
OUTPUT_CSV = "../data/raw_news_for_colab.csv"
COVERAGE_CSV = os.path.join(DATA_DIR, "news_coverage_report.csv")

DAYS_BEFORE = 14
DAYS_AFTER = 0
MAX_ARTICLES_PER_WINDOW = 10

# Premium tier: 75 requests/min, no daily cap. MAX_CALLS_PER_RUN is just a
# safety valve now (not a quota workaround) in case a bug causes runaway
# pagination -- 300 is comfortably above the ~50 calls the full backfill needs.
MAX_CALLS_PER_RUN = 300
SECONDS_BETWEEN_CALLS = 1

os.makedirs(CACHE_DIR, exist_ok=True)


In [3]:
df_earnings = pd.read_csv(BASE_CSV)
df_earnings['earnings_date'] = pd.to_datetime(df_earnings['earnings_date'])
print(f'Loaded {len(df_earnings)} earnings rows')
print(f'Date range: {df_earnings["earnings_date"].min().date()} → {df_earnings["earnings_date"].max().date()}')
print(f'Tickers: {sorted(df_earnings["ticker"].unique().tolist())}')

def av_timestamp(dt):
    return dt.strftime('%Y%m%dT%H%M')

# NEWS_SENTIMENT is queried per ticker over a date range rather than per
# earnings window, so fetch the widest span needed once per ticker and slice
# it into individual earnings windows locally afterward.
ticker_ranges = {}
for ticker, group in df_earnings.groupby('ticker'):
    start = group['earnings_date'].min() - timedelta(days=DAYS_BEFORE)
    end = group['earnings_date'].max() + timedelta(days=DAYS_AFTER)
    ticker_ranges[ticker] = (av_timestamp(start), av_timestamp(end))

ticker_ranges


Loaded 1190 earnings rows
Date range: 2021-07-19 → 2026-07-16
Tickers: ['ABT', 'ADI', 'AFL', 'AIZ', 'AMD', 'AMP', 'APA', 'APD', 'BAX', 'BKR', 'CL', 'CNC', 'CTSH', 'DAL', 'DLR', 'DOW', 'ED', 'EMR', 'EVRG', 'EXC', 'EXE', 'FCX', 'FICO', 'FOX', 'GEV', 'GNRC', 'GOOGL', 'GPN', 'HST', 'HSY', 'IRM', 'KEY', 'KMB', 'LEN', 'LH', 'LULU', 'LYV', 'MAR', 'MLM', 'MMM', 'MRVL', 'PEP', 'PM', 'PNW', 'PSKY', 'PSX', 'QCOM', 'REG', 'RVTY', 'SMCI', 'SO', 'STLD', 'SWKS', 'TPR', 'TRGP', 'TRMB', 'TTWO', 'TYL', 'ULTA', 'VTR']


{'ABT': ('20210708T0000', '20260716T0000'),
 'ADI': ('20210804T0000', '20260520T0000'),
 'AFL': ('20210714T0000', '20260429T0000'),
 'AIZ': ('20210720T0000', '20260505T0000'),
 'AMD': ('20210713T0000', '20260505T0000'),
 'AMP': ('20210712T0000', '20260423T0000'),
 'APA': ('20210721T0000', '20260506T0000'),
 'APD': ('20210726T0000', '20260430T0000'),
 'BAX': ('20210715T0000', '20260430T0000'),
 'BKR': ('20210707T0000', '20260423T0000'),
 'CL': ('20210716T0000', '20260501T0000'),
 'CNC': ('20210713T0000', '20260428T0000'),
 'CTSH': ('20210714T0000', '20260429T0000'),
 'DAL': ('20210929T0000', '20260709T0000'),
 'DLR': ('20210715T0000', '20260423T0000'),
 'DOW': ('20210708T0000', '20260423T0000'),
 'ED': ('20210722T0000', '20260507T0000'),
 'EMR': ('20210721T0000', '20260505T0000'),
 'EVRG': ('20210722T0000', '20260507T0000'),
 'EXC': ('20210721T0000', '20260506T0000'),
 'EXE': ('20210727T0000', '20260428T0000'),
 'FCX': ('20210708T0000', '20260423T0000'),
 'FICO': ('20210720T0000', '2026

In [4]:
def cache_path(ticker):
    return os.path.join(CACHE_DIR, f'{ticker}.json')

def load_cache(ticker, default_from):
    path = cache_path(ticker)
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return {"cursor": default_from, "complete": False, "articles": []}

def save_cache(ticker, cache):
    with open(cache_path(ticker), 'w') as f:
        json.dump(cache, f)

def advance_cursor(time_published):
    # AV timestamps look like YYYYMMDDTHHMMSS; step 1 minute past the last
    # article seen so the next page doesn't re-return it.
    dt = datetime.strptime(time_published, '%Y%m%dT%H%M%S') + timedelta(minutes=1)
    return dt.strftime('%Y%m%dT%H%M')

def fetch_page(ticker, time_from, time_to):
    resp = requests.get(AV_URL, params={
        'function': 'NEWS_SENTIMENT',
        'tickers': ticker,
        'time_from': time_from,
        'time_to': time_to,
        'sort': 'EARLIEST',
        'limit': 1000,
        'apikey': AV_KEY,
    })
    return resp.json()


In [5]:
calls_used = 0
quota_hit = False

for ticker, (range_from, range_to) in ticker_ranges.items():
    if quota_hit:
        break

    cache = load_cache(ticker, range_from)
    if cache["complete"]:
        print(f'{ticker}: already complete ({len(cache["articles"])} cached articles), skipping.')
        continue

    print(f'{ticker}: resuming from {cache["cursor"]} (have {len(cache["articles"])} articles so far)')

    while not cache["complete"]:
        if calls_used >= MAX_CALLS_PER_RUN:
            print(f'Hit MAX_CALLS_PER_RUN ({MAX_CALLS_PER_RUN}) for this run. '
                  f'Progress is checkpointed in {CACHE_DIR} — rerun this cell '
                  f'later (e.g. after the daily quota resets) to continue.')
            quota_hit = True
            break

        data = fetch_page(ticker, cache["cursor"], range_to)
        calls_used += 1

        if 'Information' in data or 'Note' in data or 'Error Message' in data:
            msg = data.get('Information') or data.get('Note') or data.get('Error Message')
            print(f'{ticker}: API limit/error hit after {calls_used} calls this run: {msg}')
            quota_hit = True
            break

        feed = data.get('feed', [])
        if not feed:
            print(f'{ticker}: no more articles from {cache["cursor"]} to {range_to}. Marking complete.')
            cache["complete"] = True
            break

        cache["articles"].extend(feed)
        last_time = feed[-1]['time_published']

        if len(feed) < 1000 or last_time >= range_to:
            cache["complete"] = True
        else:
            cache["cursor"] = advance_cursor(last_time)

        save_cache(ticker, cache)
        print(f'{ticker}: +{len(feed)} articles (total {len(cache["articles"])}), cursor now {cache.get("cursor")}')

        if not cache["complete"]:
            time.sleep(SECONDS_BETWEEN_CALLS)

    save_cache(ticker, cache)

print(f'\nUsed {calls_used} API calls this run.')
if quota_hit:
    print('Backfill incomplete — rerun this cell to resume once quota allows.')
else:
    print('All tickers fully backfilled.')


ABT: already complete (3454 cached articles), skipping.
ADI: already complete (2607 cached articles), skipping.
AFL: already complete (1077 cached articles), skipping.
AIZ: already complete (823 cached articles), skipping.
AMD: already complete (10419 cached articles), skipping.
AMP: already complete (2710 cached articles), skipping.
APA: already complete (2550 cached articles), skipping.
APD: already complete (1205 cached articles), skipping.


TimeoutError: [Errno 60] Operation timed out

In [ ]:
def parse_av_time(ts):
    return datetime.strptime(ts, '%Y%m%dT%H%M%S')

def ticker_relevance(article, ticker):
    for ts in article.get('ticker_sentiment', []):
        if ts.get('ticker') == ticker:
            return float(ts.get('relevance_score', 0.0))
    return 0.0

extracted_text_rows = []
coverage_rows = []

for idx, row in df_earnings.iterrows():
    ticker = row['ticker']
    earnings_date = row['earnings_date']

    cache = load_cache(ticker, None)
    window_from = earnings_date - timedelta(days=DAYS_BEFORE)
    window_to = earnings_date + timedelta(days=DAYS_AFTER)

    window_articles = [
        a for a in cache['articles']
        if window_from <= parse_av_time(a['time_published']) <= window_to
    ]
    window_articles.sort(key=lambda a: ticker_relevance(a, ticker), reverse=True)

    for article in window_articles[:MAX_ARTICLES_PER_WINDOW]:
        title = article.get('title', '')
        summary = article.get('summary', '')
        extracted_text_rows.append({
            'ticker': ticker,
            'earnings_date': earnings_date.date(),
            'full_text': f"{title}. {summary}",
        })

    coverage_rows.append({
        'ticker': ticker,
        'earnings_date': earnings_date.date(),
        'n_articles_found': len(window_articles),
        'n_articles_used': min(len(window_articles), MAX_ARTICLES_PER_WINDOW),
        'ticker_cache_complete': cache['complete'],
    })

df_colab_ready = pd.DataFrame(extracted_text_rows)
df_colab_ready.to_csv(OUTPUT_CSV, index=False)

df_coverage = pd.DataFrame(coverage_rows)
df_coverage.to_csv(COVERAGE_CSV, index=False)

print(f"Extraction complete: {len(df_colab_ready)} text rows across {len(df_earnings)} earnings windows.")

zero_coverage = df_coverage[df_coverage['n_articles_found'] == 0]
incomplete_tickers = sorted(df_coverage.loc[~df_coverage['ticker_cache_complete'], 'ticker'].unique().tolist())

print(f"\nWindows with ZERO articles: {len(zero_coverage)} / {len(df_coverage)}")
if len(zero_coverage):
    print(zero_coverage[['ticker', 'earnings_date']].to_string(index=False))

if incomplete_tickers:
    print(f"\n⚠️  Backfill still incomplete for: {incomplete_tickers} — "
          f"rerun the fetch cell to finish before trusting their zero-article windows.")

print(f"\nCoverage report saved to {COVERAGE_CSV}")
print(f"✅ Upload '{OUTPUT_CSV}' ({len(df_colab_ready)} text rows) to Google Colab for FinBERT scoring.")
